# Microsoft Agent Framework — Azure OpenAI (Responses API)

இந்த குறியீட்டு எடுத்துக்காட்டில், நீங்கள் **Microsoft Agent Framework (MAF)** ஐ பயன்படுத்தி **Azure OpenAI** மூலம் ஆதரிக்கப்படும் ஒரு எளிய முகவரியை **Responses API** மூலம் உருவாக்கப் போவீர்கள்.

> **மைக்ரேஷன் குறிப்பு:** இந்த எடுத்துக்காட்டு முன்பு Semantic Kernel மற்றும் GitHub Models பயன்படுத்தியதாக இருந்தது. இது Microsoft Agent Framework க்கு மாறியுள்ளதுடன், GitHub Models (பழையது, ஜூலை 2026ல் ஓய்வுபெறுகிறது) Azure OpenAI என மாற்றப்பட்டுள்ளது, இது Responses API ஐ ஆதரிக்கிறது. MAF இல் உள்ள `OpenAIChatClient` Azure OpenAI இன் நிலையான `/openai/v1/` முடிச்சைக் குறிக்கிறது மற்றும் Responses API ஐ இயல்புநிலை பயன்படுத்துகிறது.

இந்த எடுத்துக்காட்டின் நோக்கம், பின்னர் பல முகவர் மாதிரிகள் செயல்படுத்தப்படும் பிற குறியீட்டு எடுத்துக்காட்டுகளில் பயன்படுத்தப்படும் படிகளை விளக்குவதாகும்.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## தேவையான Python தொகுப்புகளை ஏற்று


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ஒரு கருவியை விவரித்தல்

Microsoft Agent Frameworkஇல், **கருவி** என்பது `@tool` மூலம் அலங்கரிக்கப்பட்ட நேரடி Python செயல்முறை ஆகும், அதனை முகவர் அழைக்க முடியும். கீழே, ஒரு கருவி நிர்ணயிக்கப்பட்டுள்ளது, அது ஒரு சீரற்ற விடுமுறை இடத்தை மீண்டும் செய்யாமல் வழங்குகிறது.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## முகவரியை உருவாக்குதல்

இங்கே, `TravelAgent` என்ற முகவரியை உருவாக்குகிறோம்.

இந்த உதாரணத்தில், நாம் மிகவும் அடிப்படையான அறிவுரைகளைப் பயன்படுத்துகிறோம். முகவரியின் நடத்தை எப்படி மாறுகிறது என்று கவனிக்க இந்த அறிவுரைகளை மாற்றிக் கொள்ளலாம்.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## முகவரியை இயக்கல்

இப்போது நாம் முகவரியை இயக்கலாம். முகவரி உரையாடலின் துலக்கங்களை நினைவில் வைத்திருக்கும் வகையில் `AgentSession`ஐ உருவாக்கினோம், பின்னர் இரண்டு `user_inputs`-ஐ அனுப்புகிறோம். முதலில் ஒரு பயணம் கேட்கப்படுகிறது; இரண்டாவது பயனர் பரிந்துரையை விரும்பவில்லை என்று சொல்கிறது மற்றும் மற்றொன்றை கேட்கிறது — முகவர் இன்று வரலாறும் `get_random_destination` கருவியையும் பயன்படுத்தி பதிலளிக்கிறார்.

இந்த செய்திகளை நீங்கள் மாற்றி முகவர் எப்படி வேறுபடுவதை பார்க்கலாம். பதில்கள் **token-by-token** முறையில் ஒலிக்கப்படுகின்றன.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
